# Projeto: Plataforma Semi-Submersível — Abordagem Orientada a Objetos

Modelo de otimização de plataforma semi-submersível com dois pontoons paralelos.
A análise é organizada em torno da classe `SemiSub`, que encapsula a geometria
e os métodos de cálculo hidrostático, de pesos e hidrodinâmico (RAO).

A busca pelo design ótimo (menor peso do casco sujeito a restrições) é feita via
**Evolução Diferencial** (`scipy.optimize.differential_evolution`).

## 1. Configuração

In [ ]:
import math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from typing import list, dict, Optional

pi = math.pi
matplotlib.style.use('default')

In [ ]:
class Const:
    rho_agua = 1.025    # ton/m³
    g = 9.81            # m/s²
    n_pontoons = 2
    n_colunas = 4
    gamma = 0.270

## 2. Curva de Produção

In [ ]:
def curva_prod():
    dados = [
        0.00, 120808.29, 179129.53, 197875.65, 199958.55,
        197875.65, 185378.24, 174963.73, 156217.62, 139346.11,
        121016.58, 104145.08, 85398.96, 70818.65, 54155.44,
        39575.13, 27077.72, 16663.21, 8331.61, 3749.22, 416.58,
    ]
    x = np.arange(len(dados))

    plt.figure(figsize=(12, 7))
    plt.plot(
        x, dados,
        marker='o', linestyle='-', linewidth=2, markersize=6,
        color='blue', markerfacecolor='blue', markeredgecolor='black',
    )
    plt.title('Curva de produção — bloco OML-180', fontsize=14, fontweight='bold')
    plt.xlabel('Ano', fontsize=12)
    plt.ylabel('BPD', fontsize=12)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.xticks(x)
    for i, valor in enumerate(dados):
        plt.annotate(
            f'{valor:,.2f}', (i, valor),
            textcoords='offset points', xytext=(0, 10),
            ha='center', fontsize=8,
        )
    plt.tight_layout()
    plt.show()


curva_prod()

## 3. Classe `SemiSub`

A classe centraliza geometria e análise da plataforma. Os seis parâmetros
dimensionais são armazenados no construtor e reutilizados por todos os métodos.

Os métodos são definidos como funções de módulo e depois atribuídos à classe
(`SemiSub.método = função`), permitindo separar cada método em sua própria célula
com seção de markdown correspondente.

### 3.1 Construtor

| Parâmetro | Descrição |
|-----------|----------|
| `LP` | Comprimento dos pontoons (m) |
| `BP` | Largura dos pontoons (m) |
| `HP` | Altura dos pontoons (m) |
| `BC` | Largura das colunas (m) |
| `HC` | Altura das colunas (m) |
| `T`  | Calado (m) |

In [ ]:
class SemiSub:

    def __init__(self, LP: float, BP: float, HP: float, BC: float, HC: float, T: float) -> None:
        self.LP = LP
        self.BP = BP
        self.HP = HP
        self.BC = BC
        self.HC = HC
        self.T = T

    def __repr__(self) -> str:
        return (
            f"SemiSub(LP={self.LP:.2f}, BP={self.BP:.2f}, HP={self.HP:.2f}, "
            f"BC={self.BC:.2f}, HC={self.HC:.2f}, T={self.T:.2f})"
        )

### 3.2 Hidrostática

Calcula volume deslocado $V$, altura do centro de carena $KB$, raio
metacêntrico transversal $BM$ e altura metacêntrica $KM$.

In [ ]:
def hidrostat(self, visual: int = 0):
    """
    Retorna (KM, D) — altura metacêntrica (m) e deslocamento em massa (ton).
    """
    lp, bp, hp, bc, hc, t = self.LP, self.BP, self.HP, self.BC, self.HC, self.T

    V = (4 * (t - hp) * bc**2) + (2 * lp * hp * bp)

    numerador_KB = ((lp * bp * hp**2) / 2) + (t + hp) * (t - hp) * bc**2
    denominador_KB = (lp * hp * bp) + 2 * (t - hp) * bc**2
    KB = numerador_KB / denominador_KB

    Ic = (bc**4) / 12 + bc**2 * ((lp / 2) - (bp / 2)) ** 2
    BM = 4 * Ic / V
    KM = KB + BM

    D = V * Const.rho_agua

    if bool(visual):
        print("PARÂMETROS DE ENTRADA")
        print(f"  Comprimento dos pontoons (Lp): {lp:.2f} m")
        print(f"  Largura dos pontoons (Bp):     {bp:.2f} m")
        print(f"  Altura dos pontoons (Hp):      {hp:.2f} m")
        print(f"  Largura das colunas (Bc):      {bc:.2f} m")
        print(f"  Altura das colunas (Hc):       {hc:.2f} m")
        print(f"  Calado (T):                    {t:.2f} m")
        print()
        print("PARÂMETROS HIDROSTÁTICOS:")
        print(f"  Volume deslocado: {V:.2f} m³")
        print(f"  KB: {KB:.2f} m")
        print(f"  BM: {BM:.2f} m")
        print(f"  KM: {KM:.2f} m")
        print(f"  Deslocamento em massa: {D:.2f} t")
        print()

    return KM, D


SemiSub.hidrostat = hidrostat

### 3.3 Pesos e Estabilidade

Calcula os pesos dos componentes (topside, casco, lastro, risers, amarração),
o centro de gravidade $KG$ e a altura metacêntrica $GM$.

In [ ]:
def pesos(self, boe: float = 1.2e5, visual: int = 0) -> Dict:
    """
    Retorna dict com: A_topside, P_casco, restr_lastro, GM, airgap.
    """
    lp, bp, hp, bc, hc, t = self.LP, self.BP, self.HP, self.BC, self.HC, self.T

    KM, D = self.hidrostat()

    # Topside
    A_topside = 42.246 * boe / 1e3 + 4850.1
    P_topside = boe * 83 / 1e3
    VCG_topside = hp + hc + 10

    # Casco
    V_pontoons = lp * bp * hp * Const.n_pontoons
    P_pontoons = V_pontoons * Const.gamma
    P_colunas = bc**2 * hc * Const.n_colunas * Const.gamma
    P_casco = P_pontoons + P_colunas
    VCG_casco = (
        P_pontoons * (hp / 2) + P_colunas * (hp + hc / 2)
    ) / P_casco

    # Estruturas adicionais
    P_conves = P_topside * 0.35
    P_operacional = P_topside * 0.35
    P_topside_group = P_topside + P_conves + P_operacional

    # Risers e amarração
    P_plataforma_inicial = P_topside_group + P_casco
    P_riser = P_plataforma_inicial * 0.04
    P_amarracao = P_plataforma_inicial * 0.03
    VCG_riser = hp + hc
    P_plataforma = P_plataforma_inicial + P_riser + P_amarracao

    # Lastro
    P_lastro = D - P_plataforma
    V_lastro = P_lastro / Const.rho_agua
    restr_lastro = 0 <= V_lastro <= 0.9 * V_pontoons
    VCG_lastro = hp / 2

    # KG e GM
    momento_topside = P_topside_group * VCG_topside
    momento_casco = P_casco * VCG_casco
    momento_lastro = P_lastro * VCG_lastro
    momento_riser_group = (P_riser + P_amarracao) * VCG_riser
    KG_plataforma = (
        momento_topside + momento_casco + momento_lastro + momento_riser_group
    ) / P_plataforma

    GM = KM - KG_plataforma
    airgap = hp + hc - t

    if bool(visual):
        print("TOPSIDE:")
        print(f"  Peso total topside: {P_topside_group:.1f} ton")
        print(f"  VCG topside: {VCG_topside:.2f} m")
        print(f"  Área do topside/deck: {A_topside:.1f} m²")
        print()
        print("ESTRUTURA DO CASCO:")
        print(f"  Peso estrutura casco: {P_casco:.1f} ton")
        print(f"  VCG estrutura casco: {VCG_casco:.2f} m")
        print(f"  Peso riser: {P_riser:.1f} ton")
        print(f"  Peso amarração: {P_amarracao:.1f} ton")
        print()
        print("PESO PLATAFORMA:")
        print(f"  Peso total da plataforma: {P_plataforma:,.1f} ton")
        print(f"  KG plataforma: {KG_plataforma:.2f} m")
        print()
        print("PESO LASTRO:")
        print(f"  Deslocamento: {D:.1f} ton")
        print(f"  Peso lastro: {P_lastro:.1f} ton")
        print(f"  VCG lastro: {VCG_lastro:.2f} m")
        print()
        print("ESTABILIDADE E SEGURANÇA:")
        print(f"  Airgap: {airgap:.1f} m")
        print(f"  GM: {GM:.1f} m")
        print()

    return {
        'A_topside': A_topside,
        'P_casco': P_casco,
        'restr_lastro': restr_lastro,
        'GM': GM,
        'airgap': airgap,
    }


SemiSub.pesos = pesos

### 3.4 Hidrodinâmica e RAO

Calcula as forças de Froude-Krylov nos pontoons (partes descobertas e cobertas),
a força de massa adicional, o RAO de heave e o cruzamento espectral com o espectro
JONSWAP para estimativa da resposta em mar irregular.

$$
H_{\text{heave}}(\omega) =
\frac{|F_{\text{total}}(\omega)|/\rho g A_{WF}}
{\sqrt{\left(1 - \beta^2\right)^2 + \left(2\zeta\beta\right)^2}},
\quad \beta = \omega/\omega_n
$$

In [ ]:
def rao(self, ger: int = 0, zeta: float = 0.05, visual: int = 0):
    """
    Retorna (Tn, airgap_calc, Hs_resp).

    Tn           — período natural de heave (s)
    airgap_calc  — soma da amplitude máxima da onda e do heave (m)
    Hs_resp      — altura significativa de resposta em heave (m)
    """
    lp, bp, hp, bc, hc, t = self.LP, self.BP, self.HP, self.BC, self.HC, self.T

    dp = lp - bp
    ld = lp - 2 * bc
    D = ((4 * (t - hp) * bc**2) + (2 * lp * hp * bp)) * Const.rho_agua

    # --- Forças de Froude-Krylov e massa adicional ---
    def F_FKD(w):
        A = 1
        return (
            4 * ld * Const.rho_agua * Const.g**2 * A
            * math.exp(-w**2 * t / Const.g)
            * (math.sin(w**2 * bp / (2 * Const.g)) / w**2)
            * (1 - math.exp(w**2 * hp / Const.g))
            * math.cos(w**2 * dp / (2 * Const.g))
        )

    def F_FKC(w):
        A = 1
        return (
            8 * bc * Const.rho_agua * Const.g**2 * A
            * math.exp(-w**2 * t / Const.g)
            * (math.sin(w**2 * bc / (2 * Const.g)) / w**2)
            * math.cos(w**2 * dp / (2 * Const.g))
        )

    def F_Ma(w):
        A = 1
        Ma_2D = 1.51 * Const.rho_agua * pi * (bp / 2) ** 2
        Ma = 2 * Ma_2D * ld
        forca = (
            (-2 * Ma_2D * A * ld * w**2)
            * math.exp(-w**2 * (t - hp / 2) / Const.g)
            * math.cos(w**2 * dp / (2 * Const.g))
        )
        return forca, Ma

    def F_total(w):
        return F_FKD(w) + F_FKC(w) + F_Ma(w)[0]

    Ma = F_Ma(1e-4)[1]
    wn = math.sqrt((4 * Const.rho_agua * Const.g * bc**2) / (D + Ma))
    Tn = 2 * pi / wn

    # --- Vetores de frequência e forças ---
    freq = np.linspace(1e-4, pi / 2, 1000)
    F_FKD_vals = [F_FKD(w) for w in freq]
    F_FKC_vals = [F_FKC(w) for w in freq]
    F_Ma_vals = [F_Ma(w)[0] for w in freq]
    F_total_vals = [F_total(w) for w in freq]
    F_total_RAO = np.array(F_total_vals)

    # --- RAO de heave ---
    def Z(w):
        f = F_total_RAO / (4 * Const.rho_agua * Const.g * bc**2)
        return np.abs(f) / np.sqrt((1 - (w / wn) ** 2) ** 2 + (2 * zeta * (w / wn)) ** 2)

    RAO_heave = Z(freq)

    # --- Espectro JONSWAP ---
    Tp = 14.7
    wp = 2 * pi / Tp
    Hs = 6.4
    gamma_j = 3.3
    alpha = 0.0624 / (0.23 + 0.0336 * gamma_j - 0.185 / (1.9 + gamma_j))

    def jonswap(w):
        tal = 0.07 if w <= wp else 0.09
        return (
            alpha * Hs**2 * w**(-5) / wp**(-4)
            * np.exp(-1.25 * (w / wp) ** -4)
            * gamma_j ** np.exp(-((w - wp) ** 2) / (2 * (tal * wp) ** 2))
        )

    S = np.array([jonswap(w) for w in freq])

    # --- Cruzamento espectral ---
    Sw_resp = S * RAO_heave**2
    m0_val = np.trapezoid(Sw_resp, freq)
    Hs_resp = 4 * np.sqrt(m0_val)

    A_max_onda = 1.86 * Hs / 2
    A_max_heave = 1.86 * Hs_resp / 2
    airgap_calc = A_max_onda + A_max_heave

    if bool(visual):
        print('PARÂMETROS HIDRODINÂMICOS:')
        print(f'  Massa adicional de heave: {Ma:.2f} ton')
        print(f'  Frequência natural de heave: {wn:.4f} rad/s')
        print(f'  Período natural de heave: {Tn:.2f} s')
        print(f'  Altura significativa de resposta em heave: {Hs_resp:.2f} m')
        print(f'  Amplitude significativa de resposta em heave: {Hs_resp / 2:.2f} m')

        if visual > 1:
            if ger == 0:
                plt.figure(figsize=(9, 6), dpi=150)
                plt.plot(freq, S, 'b')
                plt.title('Espectro de Energia — JONSWAP')
                plt.xlabel('Frequência (rad/s)')
                plt.ylabel('Espectro de Energia S(ω) (m²s)')
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

            plt.figure(figsize=(10, 6), dpi=150)
            plt.plot(freq, F_FKD_vals, linewidth=1.5, c='y', label='FK — Partes Descobertas')
            plt.plot(freq, F_FKC_vals, linewidth=1.5, c='g', label='FK — Partes Cobertas')
            plt.plot(freq, F_Ma_vals, linewidth=1.5, c='m', label='Força de Massa Adicional')
            plt.plot(freq, F_total_vals, linewidth=3, c='r', label='Força Total de Heave')
            plt.title('Forças Hidrodinâmicas em Heave')
            plt.legend()
            plt.xlabel('Frequência (rad/s)')
            plt.ylabel('Força (N)')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

            plt.figure(figsize=(10, 6), dpi=150)
            plt.plot(freq, RAO_heave, 'c')
            plt.plot([wn, wn], [0, 2], 'r--', label=f'ωn = {wn:.3f} rad/s')
            plt.ylim(0, 2)
            plt.title('RAO de Heave')
            plt.xlabel('Frequência (rad/s)')
            plt.ylabel('RAO (m/m)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

    return Tn, airgap_calc, Hs_resp


SemiSub.rao = rao

### 3.5 Avaliação Integrada

Combina as três análises em sequência e exibe o resultado consolidado.

In [ ]:
def avaliar(self, ger: int = 0, visual: int = 0) -> None:
    """
    Executa hidrostat(), pesos() e rao() em sequência.
    """
    self.hidrostat(visual=visual)
    self.pesos(visual=visual)
    self.rao(ger=ger, visual=visual)


SemiSub.avaliar = avaliar

## 4. Otimização por Evolução Diferencial

A busca do design ótimo usa `scipy.optimize.differential_evolution`, um algoritmo
evolutivo estocástico robusto para problemas não-lineares com restrições implícitas.
Cada geração restringe a busca a uma vizinhança ao redor do melhor design da
geração anterior, com raio proporcional a $1/(g+1)$.

### 4.1 Função Objetivo e Otimizador

**Objetivo**: minimizar o peso do casco $P_{\text{casco}}$.

**Restrições** (violações retornam penalidade $10^8$):
1. $1 \le GM \le 4$ m  
2. $T_n > 20$ s  
3. Volume do lastro $\le 90\%$ do volume dos pontoons  
4. Airgap estrutural $>$ airgap mínimo calculado pelo RAO  
5. $H_p < T < H_p + H_c$  
6. Área do deck $\le L_p^2$

In [ ]:
def otimizar(
    base: SemiSub,
    ger: int,
    maxiter: int = 300,
    popsize: int = 15,
) -> Optional[SemiSub]:
    """
    Encontra o design de menor peso do casco dentro de uma vizinhança
    ao redor de `base`, usando Evolução Diferencial.

    Retorna o melhor SemiSub viável, ou None se nenhum for encontrado.
    """
    amplitude = max(0.05, 1.0 / (ger + 1))

    lp0, bp0, hp0, bc0, hc0 = base.LP, base.BP, base.HP, base.BC, base.HC

    lp_lo, lp_hi = lp0 * (1 - amplitude), lp0 * (1 + amplitude)
    bp_lo, bp_hi = bp0 * (1 - amplitude), bp0 * (1 + amplitude)
    hp_lo, hp_hi = hp0 * (1 - amplitude), hp0 * (1 + amplitude)
    hc_lo, hc_hi = hc0 * (1 - amplitude), hc0 * (1 + amplitude)
    t_lo = hp_hi + 1            # T deve superar o maior hp esperado
    t_hi = hp_lo + hc_lo - 1    # T deve ficar abaixo do menor hp+hc esperado

    if t_lo >= t_hi:
        print(f"  Geração {ger}: faixa de calado inválida ({t_lo:.1f} >= {t_hi:.1f}).")
        return None

    bounds = [
        (lp_lo, lp_hi),
        (bp_lo, bp_hi),
        (hp_lo, hp_hi),
        (0.75 * bp_lo, bp_hi),  # BC: entre 0.75·BP e BP
        (hc_lo, hc_hi),
        (t_lo, t_hi),
    ]

    def objetivo(x):
        lp, bp, hp, bc, hc, t = x
        if not (hp < t < hp + hc):
            return 1e8
        plat = SemiSub(lp, bp, hp, bc, hc, t)
        try:
            res_p = plat.pesos()
            Tn, airgap_min, _ = plat.rao(ger=ger)
        except Exception:
            return 1e8
        if not (
            1 <= res_p['GM'] <= 4
            and Tn > 20
            and res_p['restr_lastro']
            and res_p['airgap'] > airgap_min
            and res_p['A_topside'] <= lp**2
        ):
            return 1e8
        return res_p['P_casco']

    resultado = differential_evolution(
        objetivo,
        bounds,
        maxiter=maxiter,
        popsize=popsize,
        seed=42,
        tol=1e-4,
        mutation=(0.5, 1.0),
        recombination=0.7,
        polish=True,
    )

    if resultado.fun >= 1e8:
        print(f"  Geração {ger}: nenhum design viável encontrado.")
        return None

    melhor = SemiSub(*resultado.x)
    print(f"  Geração {ger}: melhor peso do casco = {resultado.fun:,.1f} ton | {melhor}")
    return melhor

### 4.2 Loop Principal

A cada geração, `otimizar()` busca o design de menor peso ao redor do design atual.
O melhor encontrado torna-se o centro da busca na geração seguinte.

In [ ]:
def main():
    geracoes = 5
    imprimir = 2

    designAtual = SemiSub(LP=170, BP=40, HP=20, BC=23, HC=60, T=35)

    print("=" * 60)
    print("Geração 0 — Design de Referência")
    print(designAtual)
    print("=" * 60)
    designAtual.avaliar(ger=0, visual=imprimir)

    for g in range(1, geracoes + 1):
        print("\n" + "=" * 60)
        print(f"Geração {g} — Evolução Diferencial")
        print("=" * 60)
        melhor = otimizar(designAtual, ger=g, maxiter=300, popsize=15)
        if melhor is None:
            print(f"Otimização interrompida na geração {g}.")
            break
        designAtual = melhor
        designAtual.avaliar(ger=g, visual=imprimir)

    print("\n" + "=" * 60)
    print("Design final otimizado:")
    print(designAtual)
    print("=" * 60)
    designAtual.avaliar(ger=-1, visual=2)

## 5. Execução

In [ ]:
main()